***Title: Generative AI Assignment***

***Name: Gedion Sang***

***ID: CS-DA03-26010***

***Date: 08/04/2026***

## Generative AI

### Introduction
Apply understanding of Generative AI and Retrieval-Augmented Generation (RAG) to build a practical pipeline that retrieves relevant document chunks and generates context-aware answers: 
I will demonstrate your ability to:

- Apply generative AI concepts to synthesize answers from retrieved document content.
- Extract key information from a PDF document by splitting it into chunks, embedding it using Sentence-Transformers, and storing it in a FAISS vector database for efficient retrieval.
- Demonstrate how retrieval improves generative question-answering by comparing answers generated from document-grounded context versus generic answers.
- Implement a complete RAG pipeline in Python using LangChain, Hugging Face Transformers, and FAISS.
- Practice prompt engineering to structure queries that guide the generative model for clearer and more detailed responses.

### Importing Libraries

In [1]:
# 1. Install required libraries
!pip install -q langchain langchain-community langchain-huggingface transformers sentence-transformers faiss-cpu pypdf
import re
import re
import torch
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

### Step 1

In [2]:
# --- STEP 1: DEFINE CLEANING FUNCTION ---
# Fixes common PDF extraction errors like 'C y b e r' -> 'Cyber'
def fix_pdf_text(text):
    text = re.sub(r'(?<= )([^ ]) (?=[^ ])', r'\1', text)
    text = re.sub(r'^([^ ]) (?=[^ ])', r'\1', text)
    text = re.sub(r'\n([^ ]) ', r'\n\1', text)
    return re.sub(r'\s+', ' ', text).strip()


In [3]:
# --- STEP 2: UPLOAD AND PROCESS PDF ---
print('Please upload your PDF file:')
uploaded = files.upload()
if uploaded:
    pdf_path = list(uploaded.keys())[0]
    
    # Load PDF and apply cleaning
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()
    for page in pages:
        page.page_content = fix_pdf_text(page.page_content)
    # Split into chunks for retrieval
    splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)
    chunks = splitter.split_documents(pages)

Please upload your PDF file:


KeyboardInterrupt: 

In [ ]:
# --- STEP 3: CREATE VECTOR STORE ---
print('Creating embeddings and vector store...')
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

In [ ]:
 # --- STEP 4: LOAD LLM (FLAN-T5-LARGE) MANUALLY ---
    # This bypasses pipeline 'task' errors
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = 'google/flan-t5-large'
print(f'Loading {model_id} on {device}...')

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

In [ ]:
# --- STEP 5: RAG QUERY FUNCTION ---
def query_rag(question):
    relevant_docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in relevant_docs])
    # Standardized prompt for T5
    # Standardized prompt for T5
    input_text = f"answer: {question} context: {context}"
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=1024).to(device)
    # Generate response using the model directly
    # Generate response using the model directly
    outputs = model.generate(
        **inputs, 
        max_new_tokens=512, 
        do_sample=False
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# --- STEP 6: EXECUTION ---
print(f'\n--- RAG Analysis Output for {pdf_path} ---')
user_query = "Summarize the key information, professional experience, and skills mentioned in this document."
print(query_rag(user_query))
else:
print('No file uploaded.')